In [42]:
import pyvisa
import time

# === Configuration ===
channel = 1
voltage = 3.3
current_limit = 0.2
output_duration = 5  # seconds

try:
    # Initialize VISA connection
    rm = pyvisa.ResourceManager()
    instrument = rm.open_resource("USB0::0x2A8D::0x9201::MY61391394::0::INSTR")  # Replace with your device address
    print("Connected to:", instrument.query("*IDN?").strip())
    # Make sure nothing is blocking the UI
    instrument.write(":DISP:WIND:TEXT:CLE")
    
    # Configure output
    instrument.write(f"SOUR{channel}:FUNC VOLT")
    instrument.write(f"SOUR{channel}:VOLT {voltage}")
    instrument.write(f"SOUR{channel}:CURR {current_limit}")

    # Turn on output
    instrument.write(f"OUTP{channel} ON")
    time.sleep(1)
    
    # Read back values
    set_voltage = instrument.query(f"SOUR{channel}:VOLT?").strip()
    set_current = instrument.query(f"SOUR{channel}:CURR?").strip()
    meas_voltage = instrument.query(f"MEAS:VOLT? (@{channel})").strip()
    meas_current = instrument.query(f"MEAS:CURR? (@{channel})").strip()

    # Format for display
    
    display_text = (
        f"Vset={float(set_voltage):.5f}V Iset={float(set_current):.2f}A\n"
        f"Vout={float(meas_voltage):.5f}V Iout={float(meas_current):.2f}A"
    )

    # Clear screen and display text
    #instrument.write(":DISP:WIND:TEXT:CLE")
    #instrument.write(f':DISP:WIND:TEXT "{display_text}"')

    print("Displayed:\n" + display_text)
    
    # Hold for a while before turning off
    time.sleep(output_duration)

finally:
    try:
        instrument.write(f"OUTP{channel} OFF")
        instrument.write(":DISP:WIND:TEXT:CLE")
        instrument.close()
        print("Output OFF and connection closed.")
    except Exception as e:
        print("Shutdown error:", e)


Connected to: Keysight Technologies,B2902B,MY61391394,5.0.2029.1911
Displayed:
Vset=3.30000V Iset=0.20A
Vout=3.30001V Iout=0.00A
Output OFF and connection closed.
